In [0]:
%sql

select *
FROM silver.sf_salaries.silver
WHERE IdEmployee = '66ef065b11d1cb5ce483d413a16d6ec3'  
GROUP BY ALL

In [0]:
%sql

SELECT *
FROM features_store.sf_salaries.features_salaries
WHERE id_funcionario = '66ef065b11d1cb5ce483d413a16d6ec3'  
ORDER BY dt_ref

In [0]:
%pip install databricks-feature-engineering

In [0]:

dbutils.library.restartPython() 

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient 
from databricks.feature_engineering import FeatureEngineeringClient
from dateutil.relativedelta import relativedelta


In [0]:
fe = FeatureEngineeringClient()

# Ajustando ABT na camada Gold
catalog = 'gold'
database = 'sf_salaries'
table = 'abt_salario_regressao'
table_name = f'{catalog}.{database}.{table}' 
query_name = 'target_salaries'
primary_keys = ['dt_ref', 'id_funcionario']
partition_by = 'dt_ref'

# Abre o seu arquivo SQL consolidado que cria a target
with open(f"./{query_name}.sql", "r") as file_open:
    query = file_open.read()

def tables_exists(spark, catalog, database, table):
    count = (spark.sql(f"SHOW TABLES FROM {catalog}.{database}")
                    .filter(f"tableName = '{table}'")
                    .count())     
    return count == 1


In [0]:
# Criar a tabela do zero na Feature Store
if not tables_exists(spark, catalog, database, table):
    
    # Executa a query direta do arquivo SQL
    df = spark.sql(query)
    
    fe.create_table(
                    df=df,
                    name=table_name,
                    primary_keys=primary_keys,
                    partition_columns=partition_by
                        )
    print(f"ABT '{table_name}' criada com sucesso!")

# Atualiza os dados utilizando o merge da Feature Store
else:
    print(f"A tabela '{table_name}' já existe. Atualizando os dados...")
    
    # Executa a query para capturar os dados atualizados do arquivo SQL
    df = spark.sql(query) 
    
    # Atualiza os dados da tabela da ABT fazendo o merge pelas chaves primárias
    fe.write_table(df=df, name=table_name, mode='merge')
    print(f"Dados da ABT '{table_name}' atualizados com sucesso via Merge!")

In [0]:
%sql

SELECT *
FROM gold.sf_salaries.abt_salario_regressao
